# Compile Pre-Fight Career Stats

Build `pre_fight_career_stats.csv` using the same core idea as the old `preprocessing_and_EDA-1/2` notebooks:

- one row per fight
- pre-fight fighter histories only
- `avg_*` and `avg_opp_*` as exponentially weighted prior-fight averages with `span=3`
- old-style totals for rounds, title bouts, records, streaks, and win methods
- no stance columns and no weight-class columns


In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd


REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "data_clean").exists():
    REPO_ROOT = REPO_ROOT.parent

DATA_CLEAN_DIR = REPO_ROOT / "data_clean"
DATA_RAW_DIR = REPO_ROOT / "data_raw"

input_path = DATA_RAW_DIR / "ufc_fights.csv"
if not input_path.exists():
    input_path = DATA_CLEAN_DIR / "all_fights.csv"

output_path = DATA_CLEAN_DIR / "pre_fight_career_stats.csv"

raw_fights = pd.read_csv(input_path)
raw_fights["fight_id"] = np.arange(len(raw_fights))
raw_fights["date"] = pd.to_datetime(raw_fights["date"], errors="coerce")

print("Loaded:", input_path)
print("Shape:", raw_fights.shape)
raw_fights.head()


In [ ]:
# Helpers matching the old preprocessing conventions as closely as possible.

OLD_WIN_BY_COLUMNS = [
    "win_by_Decision_Majority",
    "win_by_Decision_Split",
    "win_by_Decision_Unanimous",
    "win_by_KO/TKO",
    "win_by_Submission",
    "win_by_TKO_Doctor_Stoppage",
]


def time_to_seconds(value):
    if pd.isna(value):
        return np.nan
    value = str(value).strip()
    if value == "" or value.lower() == "nan":
        return np.nan
    parts = value.split(":")
    if len(parts) != 2:
        return np.nan
    try:
        return int(parts[0]) * 60 + int(parts[1])
    except ValueError:
        return np.nan


def height_to_cm(value):
    if pd.isna(value):
        return np.nan
    value = str(value).strip()
    if value == "" or value.lower() == "nan" or value == "--":
        return np.nan
    try:
        if "'" in value:
            feet, inches = value.split("'", 1)
            inches = inches.replace('"', "").strip()
            return (int(feet) * 12 + int(inches)) * 2.54
        return float(value.replace('"', "")) * 2.54
    except ValueError:
        return np.nan


def reach_to_cm(value):
    if pd.isna(value):
        return np.nan
    value = str(value).strip()
    if value == "" or value.lower() == "nan" or value == "--":
        return np.nan
    try:
        return float(value.replace('"', "")) * 2.54
    except ValueError:
        return np.nan


def numeric_percent(value):
    if pd.isna(value):
        return np.nan
    if isinstance(value, str):
        value = value.strip()
        if value.endswith("%"):
            return pd.to_numeric(value[:-1], errors="coerce") / 100
    return pd.to_numeric(value, errors="coerce")


def scheduled_rounds(value):
    if pd.isna(value):
        return np.nan
    value = str(value)
    if "Rnd" not in value:
        return np.nan
    try:
        return int(value.split("Rnd", 1)[0].strip())
    except ValueError:
        return np.nan


def fight_time_from_round(row):
    last_round = pd.to_numeric(row.get("last_round"), errors="coerce")
    last_round_time = time_to_seconds(row.get("last_round_time"))
    if pd.isna(last_round) or pd.isna(last_round_time):
        return np.nan
    return max((last_round - 1) * 5 * 60 + last_round_time, 0)


def calculate_age_at_fight(fight_date, dob):
    dob = pd.to_datetime(dob, errors="coerce")
    if pd.isna(fight_date) or pd.isna(dob):
        return np.nan
    return (fight_date - dob).days / 365.25


def winner_side(row):
    winner = "" if pd.isna(row["winner"]) else str(row["winner"]).strip()
    if winner == row["R_fighter"]:
        return "Red"
    if winner == row["B_fighter"]:
        return "Blue"
    return "Draw"


def old_win_by_name(value):
    value = "" if pd.isna(value) else str(value).strip()
    mapping = {
        "Decision - Majority": "win_by_Decision_Majority",
        "Decision - Split": "win_by_Decision_Split",
        "Decision - Unanimous": "win_by_Decision_Unanimous",
        "KO/TKO": "win_by_KO/TKO",
        "Submission": "win_by_Submission",
        "TKO - Doctor's Stoppage": "win_by_TKO_Doctor_Stoppage",
    }
    return mapping.get(value)


def prefixed_avg_name(col):
    if col == "total_time_fought(seconds)":
        return col
    if col.startswith("opp_"):
        return col.replace("opp_", "avg_opp_", 1)
    return f"avg_{col}"


In [ ]:
# Convert each bout into Red and Blue fighter-history rows.

raw_fights["winner"] = raw_fights["winner"].fillna("").astype(str).str.strip()
raw_fights["Winner"] = raw_fights.apply(winner_side, axis=1)
raw_fights["title_bout"] = False
if "fight_type" in raw_fights.columns:
    raw_fights["title_bout"] = raw_fights["fight_type"].fillna("").astype(str).str.contains("Title Bout", case=False)
raw_fights["no_of_rounds"] = raw_fights["format"].apply(scheduled_rounds)
raw_fights["total_time_fought(seconds)"] = raw_fights.apply(fight_time_from_round, axis=1)
raw_fights["last_round"] = pd.to_numeric(raw_fights["last_round"], errors="coerce")

STAT_COLS = [
    "KD",
    "SIG_STR_pct",
    "TD_pct",
    "SUB_ATT",
    "REV",
    "SIG_STR_att",
    "SIG_STR_landed",
    "TOTAL_STR_att",
    "TOTAL_STR_landed",
    "TD_att",
    "TD_landed",
    "HEAD_att",
    "HEAD_landed",
    "BODY_att",
    "BODY_landed",
    "LEG_att",
    "LEG_landed",
    "DISTANCE_att",
    "DISTANCE_landed",
    "CLINCH_att",
    "CLINCH_landed",
    "GROUND_att",
    "GROUND_landed",
    "CTRL",
]


def build_side_rows(side):
    opp_side = "B" if side == "R" else "R"
    rows = raw_fights[[
        "fight_id",
        "date",
        "winner",
        "Winner",
        "win_by",
        "last_round",
        "title_bout",
        "total_time_fought(seconds)",
        f"{side}_fighter",
        f"{opp_side}_fighter",
        f"{side}_DOB",
        f"{side}_Height",
        f"{side}_Reach",
    ] + [f"{side}_{col}" for col in STAT_COLS] + [f"{opp_side}_{col}" for col in STAT_COLS]].copy()

    rename_cols = {
        f"{side}_fighter": "fighter_name",
        f"{opp_side}_fighter": "opponent",
        f"{side}_DOB": "DOB",
        f"{side}_Height": "Height",
        f"{side}_Reach": "Reach",
    }

    for col in STAT_COLS:
        own_name = "CTRL_time(seconds)" if col == "CTRL" else col
        opp_name = "opp_CTRL_time(seconds)" if col == "CTRL" else f"opp_{col}"
        rename_cols[f"{side}_{col}"] = own_name
        rename_cols[f"{opp_side}_{col}"] = opp_name

    rows = rows.rename(columns=rename_cols)
    rows["side"] = side
    return rows


fighter_fights = pd.concat([build_side_rows("R"), build_side_rows("B")], ignore_index=True)

fighter_fights["result"] = np.select(
    [
        fighter_fights["winner"].eq(fighter_fights["fighter_name"]),
        fighter_fights["Winner"].eq("Draw"),
    ],
    ["Win", "Draw"],
    default="Loss",
)
fighter_fights["winner_for_history"] = np.select(
    [fighter_fights["result"].eq("Win"), fighter_fights["result"].eq("Loss")],
    ["hero", "opp"],
    default="draw",
)

fighter_fights["Height_cms"] = fighter_fights["Height"].apply(height_to_cm)
fighter_fights["Reach_cms"] = fighter_fights["Reach"].apply(reach_to_cm)
fighter_fights["age"] = fighter_fights.apply(lambda row: calculate_age_at_fight(row["date"], row["DOB"]), axis=1)
fighter_fights["CTRL_time(seconds)"] = fighter_fights["CTRL_time(seconds)"].apply(time_to_seconds)
fighter_fights["opp_CTRL_time(seconds)"] = fighter_fights["opp_CTRL_time(seconds)"].apply(time_to_seconds)

for col in ["SIG_STR_pct", "TD_pct", "opp_SIG_STR_pct", "opp_TD_pct"]:
    fighter_fights[col] = fighter_fights[col].apply(numeric_percent)

for col in OLD_WIN_BY_COLUMNS:
    fighter_fights[col] = 0

winner_method_col = fighter_fights["win_by"].apply(old_win_by_name)
for col in OLD_WIN_BY_COLUMNS:
    fighter_fights[col] = ((fighter_fights["result"] == "Win") & (winner_method_col == col)).astype(int)

print("Fighter-fight shape:", fighter_fights.shape)
fighter_fights.head()


In [ ]:
# Old-style pre-fight history features.
# avg_* and avg_opp_* are EWM(span=3) averages over prior fights, not per-minute rates.

EWM_SPAN = 3

own_numeric_cols = [
    "KD",
    "SIG_STR_pct",
    "TD_pct",
    "SUB_ATT",
    "REV",
    "SIG_STR_att",
    "SIG_STR_landed",
    "TOTAL_STR_att",
    "TOTAL_STR_landed",
    "TD_att",
    "TD_landed",
    "HEAD_att",
    "HEAD_landed",
    "BODY_att",
    "BODY_landed",
    "LEG_att",
    "LEG_landed",
    "DISTANCE_att",
    "DISTANCE_landed",
    "CLINCH_att",
    "CLINCH_landed",
    "GROUND_att",
    "GROUND_landed",
    "CTRL_time(seconds)",
]

opp_numeric_cols = [f"opp_{col}" for col in own_numeric_cols if col != "CTRL_time(seconds)"] + ["opp_CTRL_time(seconds)"]
ewm_numeric_cols = own_numeric_cols + opp_numeric_cols + ["total_time_fought(seconds)"]

for col in ewm_numeric_cols + ["last_round"] + OLD_WIN_BY_COLUMNS:
    fighter_fights[col] = pd.to_numeric(fighter_fights[col], errors="coerce")

fighter_fights = fighter_fights.sort_values(["fighter_name", "date", "fight_id"]).reset_index(drop=True)

ewm_prior = fighter_fights.groupby("fighter_name", group_keys=False)[ewm_numeric_cols].apply(
    lambda group: group.ewm(span=EWM_SPAN, adjust=False).mean().shift(1)
)

prior_stats = fighter_fights[["fight_id", "fighter_name", "opponent", "date", "Height_cms", "Reach_cms", "age"]].copy()
prior_stats["total_fights"] = fighter_fights.groupby("fighter_name").cumcount()

for col in ewm_numeric_cols:
    prior_stats[prefixed_avg_name(col)] = ewm_prior[col]

prior_stats["total_rounds_fought"] = (
    fighter_fights.groupby("fighter_name")["last_round"].cumsum()
    .groupby(fighter_fights["fighter_name"])
    .shift(1)
    .fillna(0)
)
prior_stats["total_title_bouts"] = (
    fighter_fights["title_bout"].astype(int).groupby(fighter_fights["fighter_name"]).cumsum()
    .groupby(fighter_fights["fighter_name"])
    .shift(1)
    .fillna(0)
)

prior_win_by = (
    fighter_fights.groupby("fighter_name")[OLD_WIN_BY_COLUMNS].cumsum()
    .groupby(fighter_fights["fighter_name"])
    .shift(1)
    .fillna(0)
)
for col in OLD_WIN_BY_COLUMNS:
    prior_stats[col] = prior_win_by[col]


def prior_result_stats(group):
    current_win_streak = 0
    current_lose_streak = 0
    longest_win_streak = 0
    wins = 0
    losses = 0
    draw = 0
    rows = []

    for _, row in group.iterrows():
        rows.append({
            "current_win_streak": current_win_streak,
            "current_lose_streak": current_lose_streak,
            "longest_win_streak": longest_win_streak,
            "wins": wins,
            "losses": losses,
            "draw": draw,
        })

        result = row["winner_for_history"]
        if result == "hero":
            wins += 1
            current_win_streak += 1
            current_lose_streak = 0
            longest_win_streak = max(longest_win_streak, current_win_streak)
        elif result == "opp":
            losses += 1
            current_win_streak = 0
            current_lose_streak += 1
        else:
            draw += 1
            current_win_streak = 0
            current_lose_streak = 0

    return pd.DataFrame(rows, index=group.index)


result_stats = fighter_fights.groupby("fighter_name", group_keys=False).apply(prior_result_stats)
prior_stats = pd.concat([prior_stats, result_stats], axis=1)

history_feature_cols = [
    "avg_KD",
    "avg_opp_KD",
    "avg_SIG_STR_pct",
    "avg_opp_SIG_STR_pct",
    "avg_TD_pct",
    "avg_opp_TD_pct",
    "avg_SUB_ATT",
    "avg_opp_SUB_ATT",
    "avg_REV",
    "avg_opp_REV",
    "avg_SIG_STR_att",
    "avg_SIG_STR_landed",
    "avg_opp_SIG_STR_att",
    "avg_opp_SIG_STR_landed",
    "avg_TOTAL_STR_att",
    "avg_TOTAL_STR_landed",
    "avg_opp_TOTAL_STR_att",
    "avg_opp_TOTAL_STR_landed",
    "avg_TD_att",
    "avg_TD_landed",
    "avg_opp_TD_att",
    "avg_opp_TD_landed",
    "avg_HEAD_att",
    "avg_HEAD_landed",
    "avg_opp_HEAD_att",
    "avg_opp_HEAD_landed",
    "avg_BODY_att",
    "avg_BODY_landed",
    "avg_opp_BODY_att",
    "avg_opp_BODY_landed",
    "avg_LEG_att",
    "avg_LEG_landed",
    "avg_opp_LEG_att",
    "avg_opp_LEG_landed",
    "avg_DISTANCE_att",
    "avg_DISTANCE_landed",
    "avg_opp_DISTANCE_att",
    "avg_opp_DISTANCE_landed",
    "avg_CLINCH_att",
    "avg_CLINCH_landed",
    "avg_opp_CLINCH_att",
    "avg_opp_CLINCH_landed",
    "avg_GROUND_att",
    "avg_GROUND_landed",
    "avg_opp_GROUND_att",
    "avg_opp_GROUND_landed",
    "avg_CTRL_time(seconds)",
    "avg_opp_CTRL_time(seconds)",
    "total_time_fought(seconds)",
    "total_rounds_fought",
    "total_title_bouts",
    "current_win_streak",
    "current_lose_streak",
    "longest_win_streak",
    "wins",
    "losses",
    "draw",
] + OLD_WIN_BY_COLUMNS + [
    "Height_cms",
    "Reach_cms",
]

prior_stats = prior_stats[["fight_id", "fighter_name"] + history_feature_cols + ["age"]]
prior_stats.head()


In [ ]:
# Merge back to one fight-level row, then apply old-style missing value treatment.

base_cols = [
    "fight_id",
    "date",
    "R_fighter",
    "B_fighter",
    "winner",
    "Winner",
    "win_by",
    "title_bout",
]
if "fight_type" in raw_fights.columns:
    base_cols.insert(7, "fight_type")

fight_rows = raw_fights[base_cols].copy()
fight_rows["winner_side"] = fight_rows["Winner"].map({"Red": "R", "Blue": "B"}).fillna("Draw")

r_prior = prior_stats.rename(columns={
    "fighter_name": "R_fighter",
    "age": "R_age",
    **{col: f"R_{col}" for col in history_feature_cols},
})
b_prior = prior_stats.rename(columns={
    "fighter_name": "B_fighter",
    "age": "B_age",
    **{col: f"B_{col}" for col in history_feature_cols},
})

compiled_stats = fight_rows.merge(r_prior, on=["fight_id", "R_fighter"], how="left")
compiled_stats = compiled_stats.merge(b_prior, on=["fight_id", "B_fighter"], how="left")
compiled_stats = compiled_stats.sort_values(["date", "fight_id"]).reset_index(drop=True)

# The old notebook filled missing reach with height, then filled remaining numeric missing values with medians.
for side in ["R", "B"]:
    compiled_stats[f"{side}_Reach_cms"] = compiled_stats[f"{side}_Reach_cms"].fillna(compiled_stats[f"{side}_Height_cms"])

numeric_cols = compiled_stats.select_dtypes(include=[np.number]).columns
compiled_stats[numeric_cols] = compiled_stats[numeric_cols].fillna(compiled_stats[numeric_cols].median(numeric_only=True))
compiled_stats[numeric_cols] = compiled_stats[numeric_cols].fillna(0)

compiled_stats["date"] = compiled_stats["date"].dt.strftime("%Y-%m-%d")
compiled_stats["winner"] = compiled_stats["winner"].replace("", np.nan).fillna("Draw")
compiled_stats["win_by"] = compiled_stats["win_by"].replace("", np.nan).fillna("Draw")
if "fight_type" in compiled_stats.columns:
    compiled_stats["fight_type"] = compiled_stats["fight_type"].replace("", np.nan).fillna("Unknown")

metadata_cols = [col for col in [
    "fight_id",
    "date",
    "R_fighter",
    "B_fighter",
    "winner",
    "Winner",
    "winner_side",
    "win_by",
    "fight_type",
    "title_bout",
] if col in compiled_stats.columns]

final_cols = (
    metadata_cols
    + [f"B_{col}" for col in history_feature_cols]
    + [f"R_{col}" for col in history_feature_cols]
    + ["B_age", "R_age"]
)

compiled_stats = compiled_stats[final_cols]
compiled_stats.to_csv(output_path, index=False)

print("Saved:", output_path)
print("Shape:", compiled_stats.shape)
compiled_stats.head()
